In [7]:
#POC
import pydeck as pdk

# Arcs from London to several European cities
data = [
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon":  2.35,  "dst_lat": 48.85, "to": "Paris"},
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon": 13.40,  "dst_lat": 52.52, "to": "Berlin"},
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon":  9.19,  "dst_lat": 45.46, "to": "Milan"},
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon": -3.70,  "dst_lat": 40.41, "to": "Madrid"},
]

layer = pdk.Layer(
    "ArcLayer",
    data=data,
    get_source_position=["src_lon", "src_lat"],
    get_target_position=["dst_lon", "dst_lat"],
    get_source_color=[255, 0, 128],   # pink at origin
    get_target_color=[0, 200, 255],   # cyan at destination
    get_width=4,
    pickable=True,
    auto_highlight=True,
)

view_state = pdk.ViewState(
    latitude=49.0,
    longitude=4.0,
    zoom=4,
    pitch=40,
    bearing=10,
)
r = pdk.Deck(
    layers=[layer],
    initial_view_state=view_state,
    tooltip={"html": "{from} → {to}"},                       # free, no API key needed
)

# Works in VS Code / JupyterLab without any widget issues
r.show()


DeckGLWidget(carto_key=None, custom_libraries=[], google_maps_key=None, json_input='{\n  "initialViewState": {…

In [3]:
#POC
# Arcs from London to several European cities
data = [
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon":  2.35,  "dst_lat": 48.85, "to": "Paris"},
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon": 13.40,  "dst_lat": 52.52, "to": "Berlin"},
    {"from": "London",    "src_lon": -0.12,  "src_lat": 51.50, "dst_lon":  9.19,  "dst_lat": 45.46, "to": "Milan"}
]


def test(widget, payload):
    layer.data = data
    widget.update()
    print(payload)
    widget
r.deck_widget.on_click(test)
r.show()



DeckGLWidget(carto_key=None, custom_libraries=[], google_maps_key=None, json_input='{\n  "initialViewState": {…

In [27]:
# app.py
import streamlit as st
from streamlit_jupyter import StreamlitPatcher, tqdm
import pydeck as pdk
import pandas as pd

data = pd.DataFrame({
    "start_lon": [-74.006, -118.244, 151.209],
    "start_lat": [ 40.713,   34.052, -33.868],
    "end_lon":   [ -0.128,  139.692,  55.296],
    "end_lat":   [ 51.507,   35.689,  25.276],
    "label":     ["NYC → London", "LA → Tokyo", "Sydney → Dubai"],
    "passengers": [320, 280, 410],
})

# Track selected route in session state
if "selected" not in st.session_state:
    st.session_state.selected = None

def make_deck(selected_label=None):
    # Color selected arc yellow, others cyan/red
    def row_color(row):
        if selected_label and row["label"] == selected_label:
            return [255, 220, 0, 255]   # bright yellow
        return [0, 200, 255, 180]

    data["color"] = data.apply(row_color, axis=1)

    arc_layer = pdk.Layer(
        "ArcLayer",
        data=data,
        get_source_position=["start_lon", "start_lat"],
        get_target_position=["end_lon", "end_lat"],
        get_source_color="color",
        get_target_color=[255, 80, 80, 180],
        get_width=5,
        pickable=True,
        auto_highlight=True,
    )
    return pdk.Deck(
        layers=[arc_layer],
        initial_view_state=pdk.ViewState(
            latitude=20, longitude=10, zoom=1.5, pitch=40
        ),
        tooltip={"text": "{label}\n{passengers} passengers"},
        map_style="dark",
    )

# Render map and capture click events
clicked = st.pydeck_chart(
    make_deck(st.session_state.selected),
    on_select="rerun",          # re-runs the script on click
    selection_mode="single-object",
)

# Parse the click result
if clicked and clicked.selection["objects"]:
    obj = clicked.selection["objects"].get("ArcLayer", [])
    if obj:
        st.session_state.selected = obj[0]["label"]

# Sidebar info panel
if st.session_state.selected:
    row = data[data["label"] == st.session_state.selected].iloc[0]
    st.sidebar.header("✈️ Selected Route")
    st.sidebar.metric("Route", row["label"])
    st.sidebar.metric("Passengers", row["passengers"])
    st.sidebar.button("Clear selection",
                      on_click=lambda: st.session_state.update(selected=None))
else:
    st.sidebar.info("Click an arc to see details")

StreamlitPatcher().jupyter()

2026-03-01 01:33:26.175 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.176 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.179 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.180 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.181 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.181 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-03-01 01:33:26.182 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar